<a href="https://colab.research.google.com/github/68070192/sign-language-digits-cnn/blob/main/notebooks/experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import random
import shutil
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import kagglehub

# 1. ล็อก Seed เพื่อให้ผลลัพธ์รันซ้ำได้ 100% (Reproducibility)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working on device: {device}")

# 2. ดาวน์โหลด Dataset ต้นฉบับของ Arda Mavi (โหลดได้แน่นอน ไม่ติด 403)
print("Downloading Arda Mavi dataset...")
download_path = kagglehub.dataset_download("ardamavi/sign-language-digits-dataset")

# โหลดข้อมูล X, Y จากไฟล์ .npy
X = np.load(os.path.join(download_path, "X.npy"))
Y = np.load(os.path.join(download_path, "Y.npy"))
labels = np.argmax(Y, axis=1)

# 3. แตกไฟล์ Array ออกมาเป็นโฟลเดอร์รูปภาพ 0-9 อัตโนมัติ
data_dir = "sign_dataset_images"
if not os.path.exists(data_dir):
    print("Extracting images into folders 0-9...")
    for i in range(10):
        os.makedirs(os.path.join(data_dir, str(i)), exist_ok=True)

    for idx, (img_arr, label) in enumerate(zip(X, labels)):
        # แปลงเป็นภาพ RGB และเซฟลงโฟลเดอร์คลาส
        img_uint8 = (img_arr * 255).astype(np.uint8)
        img = Image.fromarray(img_uint8).convert("RGB")
        img.save(os.path.join(data_dir, str(label), f"{idx}.jpg"))
    print("Extraction complete!")
else:
    print("Image folders already exist.")

# 4. กำหนดค่าคงที่
IMG_SIZE = 64
BATCH_SIZE = 32

# 5. กำหนด Transforms (Train มี Augmentation / Val และ Test ไม่มี)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 6. โหลด Dataset ด้วย ImageFolder และแบ่ง Train (70%) / Val (15%) / Test (15%)
full_dataset_train = datasets.ImageFolder(data_dir, transform=train_transform)
full_dataset_eval = datasets.ImageFolder(data_dir, transform=eval_transform)

num_samples = len(full_dataset_train)
indices = list(range(num_samples))

np.random.seed(42)
np.random.shuffle(indices)

train_end = int(0.70 * num_samples)
val_end = int(0.85 * num_samples)

train_idx = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

train_set = Subset(full_dataset_train, train_idx)
val_set = Subset(full_dataset_eval, val_idx)
test_set = Subset(full_dataset_eval, test_idx)

# 7. สร้าง DataLoader
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n--- Data Split Summary ---")
print(f"Total Samples : {num_samples}")
print(f"Train Set     : {len(train_set)} samples ({len(train_loader)} batches)")
print(f"Validation Set: {len(val_set)} samples ({len(val_loader)} batches)")
print(f"Test Set      : {len(test_set)} samples ({len(test_loader)} batches)")
print(f"Class Mapping : {full_dataset_train.class_to_idx}")

Working on device: cuda
Using Colab cache for faster access to the 'sign-language-digits-dataset' dataset.
Extracting images into folders 0-9...
Extraction complete!

--- Data Split Summary ---
Total Samples : 2062
Train Set     : 1443 samples (46 batches)
Validation Set: 309 samples (10 batches)
Test Set      : 310 samples (10 batches)
Class Mapping : {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
